# Po2QAT Results Lab

Run a CNN, TinyViT, or TinyGPT experiment and compare the initial floating-point model with the final power-of-two model. This notebook plots task metrics, training loss, confusion matrices, and weight distributions.

[Open in Google Colab](https://colab.research.google.com/github/Ikteder/Po2QAT-Power-of-Two-Quantization-Algorithm/blob/main/notebooks/po2qat_results_lab.ipynb)

## 1. Setup

In Colab, this clones and installs the project. Locally, start Jupyter from the repository root after installing `pip install -e ".[notebook]"`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Ikteder/Po2QAT-Power-of-Two-Quantization-Algorithm.git"
REPO_DIR = Path("Po2QAT-Power-of-Two-Quantization-Algorithm")

if "COLAB_RELEASE_TAG" in os.environ:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    repo_root = next((path for path in candidates if (path / "pyproject.toml").exists()), None)
    if repo_root is None:
        raise RuntimeError("Start Jupyter from inside the cloned Po2QAT repository.")
    os.chdir(repo_root)

print("Repository:", Path.cwd())

## 2. Choose and run an experiment

Use `quick` for a short real-data classroom run or `strong` for the reference configuration. Set `RUN_EXPERIMENT = False` to revisit an existing run without retraining.

In [ ]:
MODEL = "cnn"       # cnn, vit, or llm
PROFILE = "quick"   # smoke, quick, or strong
DEVICE = "auto"     # auto, cpu, cuda, or mps
SEED = 42
RUN_EXPERIMENT = True

RUN_DIR = Path("runs") / MODEL
if RUN_EXPERIMENT:
    command = [
        sys.executable, "-m", "po2qat", "run",
        "--model", MODEL, "--profile", PROFILE,
        "--device", DEVICE, "--seed", str(SEED),
        "--output-dir", str(RUN_DIR),
    ]
    subprocess.run(command, check=True)

print("Artifacts:", RUN_DIR.resolve())

## 3. Accuracy, loss, and related task metrics

In [ ]:
import csv
import matplotlib.pyplot as plt

def read_csv(path):
    with Path(path).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

metrics = read_csv(RUN_DIR / "metrics_comparison.csv")
metric_names = ["accuracy", "loss"] if MODEL != "llm" else ["next_token_top1_accuracy", "perplexity"]
labels = ["Initial FP32", "Final Po2"]
rows = [metrics[0], metrics[-1]]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for axis, metric in zip(axes, metric_names):
    values = [float(row[metric]) for row in rows]
    bars = axis.bar(labels, values, color=["#64748b", "#2563eb"])
    axis.set_title(metric.replace("_", " ").title())
    axis.bar_label(bars, fmt="%.4f", padding=3)
    axis.spines[["top", "right"]].set_visible(False)
fig.suptitle(f"{MODEL.upper()} — initial vs final power-of-two model")
fig.tight_layout()
plt.show()

## 4. Training loss curves

In [ ]:
history = read_csv(RUN_DIR / "training_history.csv")
phases = []
for row in history:
    if row["phase"] not in phases:
        phases.append(row["phase"])

fig, axis = plt.subplots(figsize=(9, 4.5))
for phase in phases:
    phase_rows = [row for row in history if row["phase"] == phase]
    axis.plot(
        range(1, len(phase_rows) + 1),
        [float(row["train_loss"]) for row in phase_rows],
        marker="o", label=phase.replace("_", " ").title(),
    )
axis.set(xlabel="Recorded training update", ylabel="Training loss", title="Training loss by phase")
axis.legend()
axis.grid(alpha=0.2)
axis.spines[["top", "right"]].set_visible(False)
plt.show()

## 5. Confusion matrices

CNN and ViT runs save initial and final confusion matrices. Language modeling uses token-level metrics instead.

In [ ]:
if MODEL == "llm":
    print("Confusion matrices do not apply to the language-model task.")
else:
    matrices = [
        ("Initial FP32", RUN_DIR / "confusion_matrix_initial_fp32.csv"),
        ("Final Po2", RUN_DIR / "confusion_matrix_po2_quantized.csv"),
    ]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for axis, (title, path) in zip(axes, matrices):
        rows_cm = read_csv(path)
        class_columns = [column for column in rows_cm[0] if column != "actual\\predicted"]
        matrix = [[int(row[column]) for column in class_columns] for row in rows_cm]
        image = axis.imshow(matrix, cmap="Blues")
        axis.set(title=title, xlabel="Predicted class", ylabel="True class")
        fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    fig.suptitle(f"{MODEL.upper()} confusion matrices")
    fig.tight_layout()
    plt.show()

## 6. Weight distributions

The final histogram should form discrete spikes because every reported quantized weight is zero or an exact signed power of two.

In [ ]:
import torch

weight_rows = read_csv(RUN_DIR / "weight_summary.csv")
tensor_name = next(row["tensor"] for row in weight_rows if row["quantized"].lower() == "true")
checkpoints = [
    ("Initial FP32", RUN_DIR / "initial_fp32.pt"),
    ("QAT master", RUN_DIR / "qat_master_fp32.pt"),
    ("Final Po2", RUN_DIR / "po2_quantized.pt"),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, (label, path) in zip(axes, checkpoints):
    state = torch.load(path, map_location="cpu", weights_only=True)
    values = state[tensor_name].detach().flatten().numpy()
    axis.hist(values, bins=60, color="#2563eb", alpha=0.85)
    axis.set(title=label, xlabel="Weight value", ylabel="Count")
    axis.spines[["top", "right"]].set_visible(False)
fig.suptitle(f"Weight distribution: {tensor_name}")
fig.tight_layout()
plt.show()

print("Inspect sampled values in:", RUN_DIR / "weight_comparison.csv")
print("Inspect the packed exact sign/exponent arrays in:", RUN_DIR / "po2_sign_exponent.npz")

## Reflection

Use the assignment worksheet to explain the accuracy–compression trade-off, interpret the confusion matrix, and verify the exact power-of-two invariant rather than judging the model from accuracy alone.